# 📊 Telecom Customer Churn Analysis
## Notebook 1 of 2 — Exploratory Data Analysis (EDA)

---

| Item | Detail |
|------|--------|
| **Project** | Customer Churn Prediction — Telecom |
| **Dataset** | 7,043 customers × 21 features (3-table merge) |
| **Notebook** | 01 — Problem Statement → Data Loading → Merging → EDA |
| **Next Notebook** | `02_Feature_Engineering_Modeling.ipynb` |

---

In [ ]:
# --- Smart Path Resolution (works in VS Code, Jupyter, and Colab) ---
import os
from pathlib import Path

if os.path.exists('../data/raw'):
    # Running from notebooks/ directory (VS Code / Jupyter)
    DATA_RAW       = Path('../data/raw')
    DATA_PROCESSED = Path('../data/processed')
elif os.path.exists('data/raw'):
    # Running from project root
    DATA_RAW       = Path('data/raw')
    DATA_PROCESSED = Path('data/processed')
else:
    # Google Colab — mount Drive or adjust this path
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/customer-churn-analysis')
    DATA_RAW       = BASE / 'data' / 'raw'
    DATA_PROCESSED = BASE / 'data' / 'processed'

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
print(f"DATA_RAW       : {DATA_RAW.resolve()}")
print(f"DATA_PROCESSED : {DATA_PROCESSED.resolve()}")

---
## 📋 Notebook Structure (This File)

```
1. Import Libraries
2. Load Datasets (3 source tables)
3. Data Merging (3-way inner join on customerID)
4. Data Understanding
   4a. Shape & Schema
   4b. Statistical Summary
   4c. Data Type Inspection
5. Data Quality Inspection
   5a. Missing Values
   5b. Duplicate Rows
   5c. TotalCharges type anomaly
   5d. Outlier Profiling
6. Exploratory Data Analysis (EDA)
   6a. Target Distribution (Churn Balance)
   6b. Univariate — Categorical Features
   6c. Univariate — Numerical Features
   6d. Bivariate — Categoricals vs. Churn
   6e. Bivariate — Numerics vs. Churn
   6f. Correlation Heatmap
7. EDA Summary & Key Insights
```

> **Output saved**: `../data/processed/telecom_cleaned.csv` for use in Notebook 2.

---
## 1. 📦 Import Libraries

In [ ]:
from IPython.display import display
%matplotlib inline
# Core data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Settings
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)

# Plot style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")
print(f"seaborn : {sns.__version__}")

In [ ]:
# --- Smart Path Resolution (works in VS Code, Jupyter, and Colab) ---
import os
from pathlib import Path

if os.path.exists('../data/raw'):
    # Running from notebooks/ directory (VS Code / Jupyter)
    DATA_RAW       = Path('../data/raw')
    DATA_PROCESSED = Path('../data/processed')
elif os.path.exists('data/raw'):
    # Running from project root
    DATA_RAW       = Path('data/raw')
    DATA_PROCESSED = Path('data/processed')
else:
    # Google Colab — mount Drive or adjust this path
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/customer-churn-analysis')
    DATA_RAW       = BASE / 'data' / 'raw'
    DATA_PROCESSED = BASE / 'data' / 'processed'

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
print(f"DATA_RAW       : {DATA_RAW.resolve()}")
print(f"DATA_PROCESSED : {DATA_PROCESSED.resolve()}")

In [ ]:
# --- Load all three source tables ---
churn_data    = pd.read_csv(DATA_RAW / 'churn_data.csv')
customer_data = pd.read_csv(DATA_RAW / 'customer_data.csv')
internet_data = pd.read_csv(DATA_RAW / 'internet_data.csv')

print(f"churn_data    : {churn_data.shape[0]:,} rows × {churn_data.shape[1]} columns")
print(f"customer_data : {customer_data.shape[0]:,} rows × {customer_data.shape[1]} columns")
print(f"internet_data : {internet_data.shape[0]:,} rows × {internet_data.shape[1]} columns")


In [ ]:
churn_data.head()

In [ ]:
customer_data.head()

In [ ]:
internet_data.head()

---
## 3. 🔗 Data Merging (3-way inner join)

We merge all three tables on `customerID` using **inner joins** to retain only records present in all three sources.  
Inner join is appropriate here because all 7,043 IDs are present in every table (confirmed by shape checks above).

In [ ]:
# Step 1: merge churn + demographics
df1 = pd.merge(churn_data, customer_data, how='inner', on='customerID')
print(f"After merge 1 (churn + customer): {df1.shape}")

# Step 2: merge result + internet/service data
telecom = pd.merge(df1, internet_data, how='inner', on='customerID')
print(f"After merge 2 (+ internet):       {telecom.shape}")

In [ ]:
telecom.head(3)

---
## 4. 🔍 Data Understanding

### 4a. Schema & Shape

In [ ]:
print(f"Dataset dimensions: {telecom.shape[0]:,} rows × {telecom.shape[1]} columns")
print()
telecom.info()

> ⚠️ **Note**: `TotalCharges` is dtype `object` (string) even though it should be numeric. This is a known data quality issue — caused by blank/whitespace entries for customers with `tenure=0`. We will fix this in the Data Quality section.

### 4b. Statistical Summary — Numerical Features

In [ ]:
telecom.describe()

### 4c. Statistical Summary — Categorical Features

In [ ]:
telecom.describe(include='O')

### 4d. Data Dictionary Reference

In [ ]:
# Load data dictionary (with encoding fallback for robustness)
dict_file = DATA_RAW / 'Telecom_Churn_Data_Dictionary.csv'
try:
    data_dict = pd.read_csv(dict_file, encoding='utf-8')
except UnicodeDecodeError:
    data_dict = pd.read_csv(dict_file, encoding='latin1')

data_dict.columns = data_dict.columns.str.strip()
data_dict


---
## 5. 🧹 Data Quality Inspection

### 5a. Missing Value Analysis

In [ ]:
missing = pd.DataFrame({
    'Count'  : telecom.isnull().sum(),
    'Percent': round(100 * telecom.isnull().sum() / len(telecom), 2)
}).query('Count > 0')

if missing.empty:
    print("✅ No NaN missing values detected in the merged dataframe.")
else:
    display(missing)

### 5b. Duplicate Row Check

In [ ]:
dups = telecom.duplicated().sum()
print(f"Duplicate rows: {dups}")
cust_dups = telecom['customerID'].duplicated().sum()
print(f"Duplicate customerIDs: {cust_dups}")

### 5c. TotalCharges — Type Anomaly & Hidden Blanks

`TotalCharges` is stored as a string. The blank entries correspond to customers with `tenure = 0` (newly signed but not yet billed).

In [ ]:
# Count how many entries are whitespace / empty string
blank_mask = telecom['TotalCharges'].str.strip() == ''
print(f"Blank TotalCharges entries: {blank_mask.sum()}")

# Show those rows
telecom[blank_mask][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']].head(10)

In [ ]:
# Coerce to numeric — blank strings become NaN safely
telecom['TotalCharges'] = pd.to_numeric(telecom['TotalCharges'], errors='coerce')

# Drop the 11 rows with tenure=0 (they have no meaningful billing history)
telecom = telecom.dropna(subset=['TotalCharges']).copy()
telecom.reset_index(drop=True, inplace=True)

print(f"Dataset after cleaning: {telecom.shape}")

### 5d. Outlier Profiling (Percentile Analysis)

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
telecom[num_cols].describe(percentiles=[.25, .50, .75, .90, .95, .99])

> **Insight**: The percentile progression is smooth and gradual across all three numeric columns — there are **no extreme outliers** that would distort model training. The max values are well within reasonable real-world ranges for a telecom dataset.

---
## 6. 📈 Exploratory Data Analysis (EDA)

### 6a. Target Distribution — Churn Balance

In [ ]:
churn_counts = telecom['Churn'].value_counts()
churn_pct    = telecom['Churn'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count bar
axes[0].bar(['No Churn', 'Churned'], churn_counts.values, color=['#2196F3', '#F44336'], edgecolor='black')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold', fontsize=12)
axes[0].set_title('Churn Count', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(churn_counts.values, labels=['No Churn', 'Churned'],
            autopct='%1.1f%%', colors=['#2196F3', '#F44336'],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Churn Proportion', fontsize=14, fontweight='bold')

plt.suptitle('Target Variable — Churn Distribution', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Churn Rate  : {churn_pct.get('Yes', churn_pct.iloc[1]):.2f}%")
print(f"Retain Rate : {churn_pct.get('No',  churn_pct.iloc[0]):.2f}%")

> **Class Imbalance Note**: The dataset has a ~26.6% churn rate — a **mild class imbalance** (3:1 ratio). This is not extreme enough to require SMOTE or class weighting, but we will address it during model evaluation by choosing an appropriate decision threshold rather than relying on raw accuracy.

### 6b. Univariate Analysis — Categorical Features (Demographics)

In [ ]:
def plot_categorical_counts(cols, df, title, figsize=(16, 5)):
    """Plots count bars with value annotations for a list of categorical columns."""
    fig, axes = plt.subplots(1, len(cols), figsize=figsize)
    if len(cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, cols):
        order = df[col].value_counts().index
        sns.countplot(x=col, data=df, ax=ax, palette='Set2', order=order)
        for bar in ax.patches:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, h + 10, f'{int(h):,}',
                    ha='center', va='bottom', fontsize=9, fontweight='bold')
        ax.set_title(f'{col}', fontsize=12, fontweight='bold')
        ax.set_xlabel('')
        ax.set_ylabel('Count')
        ax.tick_params(axis='x', rotation=30)
    plt.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

plot_categorical_counts(
    ['gender', 'SeniorCitizen', 'Partner', 'Dependents'],
    telecom, 'Group 1 — Demographics'
)

In [ ]:
plot_categorical_counts(
    ['Contract', 'PaperlessBilling', 'PaymentMethod'],
    telecom, 'Group 2 — Contract & Billing', figsize=(16, 5)
)

In [ ]:
plot_categorical_counts(
    ['PhoneService', 'MultipleLines', 'InternetService'],
    telecom, 'Group 3 — Core Services'
)

In [ ]:
plot_categorical_counts(
    ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection'],
    telecom, 'Group 4 — Value-Added Services (Part 1)'
)

In [ ]:
plot_categorical_counts(
    ['TechSupport', 'StreamingTV', 'StreamingMovies'],
    telecom, 'Group 5 — Value-Added Services (Part 2)'
)

### 6c. Univariate Analysis — Numerical Features

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

for i, col in enumerate(num_cols):
    # Histogram
    axes[0, i].hist(telecom[col], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    axes[0, i].set_title(f'{col} — Histogram', fontweight='bold')
    axes[0, i].set_xlabel(col)
    axes[0, i].set_ylabel('Frequency')
    # Boxplot
    axes[1, i].boxplot(telecom[col].dropna(), vert=True, patch_artist=True,
                       boxprops=dict(facecolor='steelblue', alpha=0.6))
    axes[1, i].set_title(f'{col} — Boxplot', fontweight='bold')
    axes[1, i].set_ylabel(col)

plt.suptitle('Numerical Feature Distributions', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 6d. Bivariate Analysis — Categorical Features vs. Churn

We'll visualize churn rate (%) for each category to quickly identify **high-risk segments**.

In [ ]:
def churn_rate_by_category(df, col, ax, palette='coolwarm'):
    """Plots % churn rate for each category of a column."""
    churn_rate = df.groupby(col)['Churn'].apply(
        lambda x: (x == 'Yes').mean() * 100
    ).reset_index()
    churn_rate.columns = [col, 'Churn_Rate']
    churn_rate = churn_rate.sort_values('Churn_Rate', ascending=False)
    
    bars = ax.bar(churn_rate[col].astype(str), churn_rate['Churn_Rate'],
                  color=sns.color_palette(palette, len(churn_rate)), edgecolor='black')
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.5, f'{h:.1f}%',
                ha='center', va='bottom', fontsize=8, fontweight='bold')
    ax.set_title(f'Churn Rate by {col}', fontweight='bold', fontsize=11)
    ax.set_xlabel('')
    ax.set_ylabel('Churn Rate (%)')
    ax.set_ylim(0, churn_rate['Churn_Rate'].max() * 1.25)
    ax.tick_params(axis='x', rotation=25)

cat_cols_bivariate = [
    'Contract', 'InternetService', 'PaymentMethod',
    'TechSupport', 'OnlineSecurity', 'SeniorCitizen',
    'PaperlessBilling', 'MultipleLines', 'StreamingTV'
]

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(cat_cols_bivariate):
    churn_rate_by_category(telecom, col, axes[i])

plt.suptitle('Churn Rate by Categorical Feature', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 6e. Bivariate Analysis — Numerical Features vs. Churn

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, col in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges']):
    sns.boxplot(x='Churn', y=col, data=telecom, palette={'Yes': '#F44336', 'No': '#2196F3'},
                ax=ax, showfliers=True, linewidth=1.2)
    ax.set_title(f'{col} vs. Churn', fontweight='bold', fontsize=13)
    ax.set_xlabel('Churn', fontsize=11)
    ax.set_ylabel(col, fontsize=11)

plt.suptitle('Numerical Features vs. Churn (Box Plots)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

**Tenure vs. Churn Insight:**  
Churned customers have a **median tenure of ~10 months**, while loyal customers sit at ~38 months. 75% of all churn happens before the 29-month mark — the first year of service is the highest-risk zone.

**MonthlyCharges vs. Churn Insight:**  
Churned customers pay a **higher median monthly charge (~\$80)** vs. loyal customers (~\$64). Premium pricing tiers lack stickiness without matching perceived value.

**TotalCharges vs. Churn Insight:**  
Lower TotalCharges in churned customers aligns with shorter tenure — these are newer, high-spend customers who leave quickly.

In [ ]:
# KDE plot - richer view of tenure distribution split by churn
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges']):
    for label, color in [('Yes', '#F44336'), ('No', '#2196F3')]:
        subset = telecom[telecom['Churn'] == label][col]
        ax.hist(subset, bins=30, alpha=0.5, color=color, label=f'Churn={label}', density=True, edgecolor='white')
    ax.set_title(f'{col} Distribution by Churn', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('Distribution Overlap — Numerical Features by Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 6f. Correlation Heatmap (Numeric + SeniorCitizen)

In [ ]:
# Encode Churn temporarily for correlation analysis
corr_df = telecom[['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']].copy()
corr_df['Churn_num'] = (telecom['Churn'] == 'Yes').astype(int)

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_df.corr(), dtype=bool))
sns.heatmap(corr_df.corr(), annot=True, fmt='.3f', cmap='coolwarm',
            mask=mask, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix (Numeric Features + Churn)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

> **Notable correlations with Churn:**  
> - `tenure` (−0.35) — longer tenure → less churn  
> - `MonthlyCharges` (+0.19) — higher charges → more churn  
> - `TotalCharges` (−0.20) — higher total (longer tenure) → less churn  
> - `TotalCharges` & `tenure` are highly correlated (0.83) → potential multicollinearity issue to handle in Notebook 2

### 6g. Senior Citizen Deep-Dive

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count by Senior status
telecom['SeniorCitizen_Label'] = telecom['SeniorCitizen'].map({0: 'Non-Senior', 1: 'Senior'})
sns.countplot(x='SeniorCitizen_Label', hue='Churn', data=telecom,
              palette={'Yes': '#F44336', 'No': '#2196F3'}, ax=axes[0])
axes[0].set_title('Churn Count by Senior Status', fontweight='bold')
axes[0].set_xlabel('')

# Churn rate by senior status
senior_churn = telecom.groupby('SeniorCitizen_Label')['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100
).reset_index()
senior_churn.columns = ['SeniorCitizen_Label', 'Churn_Rate']
axes[1].bar(senior_churn['SeniorCitizen_Label'], senior_churn['Churn_Rate'],
            color=['#2196F3', '#F44336'], edgecolor='black')
for i, v in enumerate(senior_churn['Churn_Rate']):
    axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')
axes[1].set_title('Churn Rate by Senior Status', fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_ylim(0, 55)

plt.suptitle('Senior Citizens are Disproportionately at Risk', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Clean up helper column
telecom.drop(columns=['SeniorCitizen_Label'], inplace=True)

### 6h. Contract Type — The Strongest Predictor

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stacked proportion chart
contract_churn = telecom.groupby('Contract')['Churn'].value_counts(normalize=True).unstack() * 100
contract_churn.plot(kind='bar', stacked=True, ax=axes[0],
                    color=['#2196F3', '#F44336'], edgecolor='black')
axes[0].set_title('Churn Proportion by Contract Type', fontweight='bold')
axes[0].set_ylabel('Percentage (%)')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend(['No Churn', 'Churned'])

# Average tenure by contract
contract_tenure = telecom.groupby('Contract')['tenure'].mean().reset_index()
axes[1].bar(contract_tenure['Contract'], contract_tenure['tenure'],
            color=['#FF9800', '#4CAF50', '#9C27B0'], edgecolor='black')
for i, v in enumerate(contract_tenure['tenure']):
    axes[1].text(i, v + 0.5, f'{v:.1f}mo', ha='center', fontweight='bold')
axes[1].set_title('Average Tenure by Contract Type', fontweight='bold')
axes[1].set_ylabel('Average Tenure (months)')
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('Contract Type: The #1 Churn Predictor', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. 💾 Save Cleaned Dataset for Notebook 2

In [ ]:
# Save cleaned merged data for Notebook 2
os.makedirs(DATA_PROCESSED, exist_ok=True)
output_path = DATA_PROCESSED / 'telecom_cleaned.csv'
telecom.to_csv(output_path, index=False)
print(f"Saved {len(telecom):,} rows to: {output_path.resolve()}")


---
## 8. 📋 EDA Summary & Key Insights

| # | Insight | Business Action |
|---|---------|----------------|
| 1 | **26.6% churn rate** — mild imbalance (3:1 ratio) | Use threshold tuning, not accuracy, to evaluate model |
| 2 | **Month-to-month contract** customers churn at ~43% vs. <5% for 2-year | Offer incentives to migrate to long-term contracts |
| 3 | **Fiber Optic** users churn at ~42% — the highest of any internet tier | Audit service quality, pricing vs. DSL alternatives |
| 4 | **New customers (tenure < 12 mo)** are highest risk | Deploy early-warning retention programs |
| 5 | **Senior citizens** churn at 41% vs. 24% for non-seniors | Create tailored senior plans with dedicated support |
| 6 | **Electronic Check** payment churn ~45% — highest of all methods | Encourage autopay migration (credit card/bank transfer) |
| 7 | **TechSupport = No** → 42% churn vs. 15% with support | Bundle tech support in retention offers |
| 8 | **TotalCharges ↔ tenure** correlation = 0.83 → multicollinearity | Will drop TotalCharges or MonthlyCharges in modeling |
| 9 | No extreme outliers in numeric features | No winsorization needed before modeling |

---

**➡️ Continue to Notebook 2:** [`02_Feature_Engineering_Modeling.ipynb`](./02_Feature_Engineering_Modeling.ipynb)